# Lab 02 — Housing Price Regression — Full Workflow
**Regression Track** · Intermediate · ~75 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Preprocess mixed numeric/categorical features
2. Compare LinearRegression vs RandomForest
3. Use cross-validation for honest evaluation
4. Read feature importance for model insight

## Datasets (this folder)
- `housing.csv`

## How to run on Google Colab
1. Open this notebook from the Kongu AI Academy lab page (**Open in Colab**). Colab saves a copy to your Google Drive.
2. Run **Cell 0** first: it downloads and unzips the dataset.
3. Write your code in the exercise cells and run them.
4. Run the last cell, `kaa_check()`, and compare your answers with the expected outputs in the exercises.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`housing.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — Kongu AI Academy: download the dataset (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-02-housing-price-regression"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/abhisek8113/kongu-ai-academy/main/public/course-labs/course-02-classical-machine-learning/lab-02-housing-price-regression/bundle/dataset.zip"
NEED = ["housing.csv"]  # unzipped files used by the code below
EXERCISES = ["rf_mae_gap"]  # functions you write in the exercise cells


def kaa_check():
    """Self-check: runs each exercise function and prints what it returns."""
    for name in EXERCISES:
        fn = globals().get(name)
        if not callable(fn):
            print(f"✗ {name}: not written yet")
            continue
        try:
            print(f"✓ {name}() ->", fn())
        except TypeError:
            print(f"✓ {name} is defined (takes arguments; test it with the example in its exercise)")
        except Exception as exc:
            print(f"✗ {name} raised {type(exc).__name__}: {exc}")


def _have_files():
    return all(os.path.exists(f) for f in NEED)


if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", DATASET_ZIP_URL, "-O", "dataset.zip"], check=True)
    else:
        urllib.request.urlretrieve(DATASET_ZIP_URL, "dataset.zip")
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))

## Real Estate: Predicting California Housing Prices

> **Scenario:** You are a data science consultant for a real estate investment firm. The firm wants to predict median house values in California districts using features like location, income, and housing density. You have 20,640 census block-group records. Your task is to build and compare regression models, handle missing data, encode categorical variables, and identify the most important features.
>
> **You will learn:** train/test split, median imputation, one-hot encoding, LinearRegression vs RandomForest, cross-validation, feature importance.
> **Time:** ~75 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | What it means | Where it shows up |
|---|---|---|
| Train/test split | Hold out 20% of data for unbiased evaluation | `train_test_split(test_size=0.2)` |
| Median imputation | Fill missing values with the column median | `SimpleImputer(strategy="median")` |
| One-hot encoding | Convert categories to binary columns | `pd.get_dummies()` |
| LinearRegression | Linear model — fast, interpretable | `LinearRegression()` |
| RandomForest | Ensemble of decision trees — captures non-linearity | `RandomForestRegressor()` |
| Cross-validation | K-fold evaluation for robust estimates | `cross_val_score()` |
| Feature importance | Which features drive predictions | `rf.feature_importances_` |

---

## 1. Load and Explore the Data

The California housing dataset is a classic for a reason: it is messy enough to be realistic, clean enough to be approachable, and rich enough to teach the full ML workflow. With 20,640 census block groups and 10 columns, it is large enough that computational choices matter but small enough that every model trains in seconds.

The target is `median_house_value` — the median price of homes in that block group, in hundreds of thousands of dollars. The features tell a story about each neighbourhood: where it is (longitude, latitude), how old the homes are (housing_median_age), how big they are (total_rooms, total_bedrooms), how many people live there (population, households), how wealthy the residents are (median_income), and how close the ocean is (ocean_proximity).

Two data quality issues jump out immediately. First, `total_bedrooms` has 207 missing values — about 1% of the data. We cannot simply drop these rows; that would discard 1% of our data and potentially bias the sample. Instead, we will **impute** them: fill in reasonable values based on the rest of the data. Second, `ocean_proximity` is a categorical variable with five values (`<1H OCEAN`, `INLAND`, `NEAR OCEAN`, `NEAR BAY`, `ISLAND`). Machine learning models need numbers, so we will **encode** these categories as binary columns — a process called one-hot encoding.

The target also has a quirk: it is capped at 500,001. Any home worth more than $500,000 is recorded as exactly $500,001. This means the model cannot predict above that value — a limitation we must remember when interpreting results.

Let us load the data and see what we are working with.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"housing.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())
print(df["ocean_proximity"].value_counts())


**What to notice:**
- 20,640 rows, 10 columns (9 features + 1 target).
- `total_bedrooms` has missing values — we must impute them.
- `ocean_proximity` is categorical with 5 distinct values — needs encoding.
- `median_house_value` ranges up to 500,001 (capped).

> **Pitfall:** Never fit an imputer on the full dataset before splitting — that leaks test information into training. Fit on train only.

---

## 2. Preprocess: Impute and Encode

Before any model can learn from data, the data must be in a form the model can understand. This is the unglamorous but critical work of **preprocessing**, and it is where most real-world ML projects spend most of their time. The rule is simple: garbage in, garbage out. No model, no matter how sophisticated, can overcome badly prepared data.

Our preprocessing has two steps. First, we handle the missing `total_bedrooms` values. The safest approach is **median imputation**: fill each missing value with the median of that column. We use the median rather than the mean because it is robust to outliers — a few mansions with 50 bedrooms will not skew our imputed values. Crucially, we fit the imputer on the **training data only**, then apply it to both train and test. If we fit on the full dataset, information from the test set leaks into training — a subtle but serious form of cheating that inflates your metrics.

Second, we encode the categorical `ocean_proximity` column. One-hot encoding creates a binary column for each category: `ocean_proximity_INLAND`, `ocean_proximity_NEAR_BAY`, etc. We drop the first category to avoid the "dummy variable trap" — perfect collinearity that can confuse linear models. The `reindex` step ensures the test set has the same columns as the training set, even if some categories happen to be missing from the test split.

The result is a clean numeric matrix ready for modelling. Let us build it.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

df = pd.read_csv(r"housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_test_num = pd.DataFrame(
    imputer.transform(X_test.select_dtypes(include=[np.number])),
    columns=X_test.select_dtypes(include=[np.number]).columns,
    index=X_test.index,
)

X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_test_cat = pd.get_dummies(X_test[["ocean_proximity"]], drop_first=True)
X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)

X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_proc = pd.concat([X_test_num, X_test_cat], axis=1)

print(f"Train shape: {X_train_proc.shape}")
print(f"Test shape:  {X_test_proc.shape}")
print(f"Missing in train: {X_train_proc.isnull().sum().sum()}")
print(f"Missing in test:  {X_test_proc.isnull().sum().sum()}")


**What to notice:**
- After imputation, zero missing values remain.
- One-hot encoding adds 4 binary columns (5 categories − 1 dropped reference).
- The processed feature matrix has 13 columns (9 numeric + 4 one-hot).

> **Pitfall:** `pd.get_dummies` on train and test separately can produce different columns. Use `reindex` to align test columns with train.

---

## 3. Baseline: Mean Predictor

Before building a real model, establish a baseline. The simplest possible model always predicts the training mean. Any real model should beat this.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error

df = pd.read_csv(r"housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline_pred = np.full_like(y_test, y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline (mean) MAE: ${baseline_mae:,.2f}")


**What to notice:**
- The baseline MAE tells you how far off a naive guess would be.
- This is the number your models must beat to be useful.

> **Pitfall:** Always compute the baseline on the training mean, not the test mean — using test data for the baseline is leakage.

---

## 4. Linear Regression vs Random Forest

Here is where the lab gets interesting. We are going to fit two very different models on the same preprocessed data and see which one wins. This is not just an academic exercise — model comparison is a core skill in applied machine learning, and the choice of model can mean the difference between a project that ships and one that does not.

**Linear Regression** assumes the target is a linear combination of the features. It is fast, interpretable, and works well when the true relationship is approximately linear. But housing prices are not linear: the effect of income on price diminishes at high incomes, location interacts with size in complex ways, and the capped target creates a ceiling effect. A straight line cannot capture these nuances.

**Random Forest** takes a completely different approach. It builds hundreds of decision trees, each trained on a random subset of the data and features, then averages their predictions. Each tree asks a series of yes/no questions about the features ("Is median_income > 5.2?" "Is longitude < -118?") and arrives at a prediction. By averaging many trees, the forest captures complex non-linear relationships and feature interactions that a linear model cannot see. The cost is interpretability — a forest of 100 trees is much harder to explain than a single equation.

The metric we will use is **Mean Absolute Error (MAE)** — the average absolute difference between predicted and actual prices. Unlike MSE, MAE is in the same units as the target (dollars), making it directly interpretable. An MAE of $30,000 means the model's predictions are typically about $30,000 off.

Let us fit both models and compare.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv(r"housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_test_num = pd.DataFrame(
    imputer.transform(X_test.select_dtypes(include=[np.number])),
    columns=X_test.select_dtypes(include=[np.number]).columns,
    index=X_test.index,
)
X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_test_cat = pd.get_dummies(X_test[["ocean_proximity"]], drop_first=True)
X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)
X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)
X_test_proc = pd.concat([X_test_num, X_test_cat], axis=1)

lr = LinearRegression()
lr.fit(X_train_proc, y_train)
lr_pred = lr.predict(X_test_proc)
lr_mae = mean_absolute_error(y_test, lr_pred)
print(f"Linear Regression MAE: ${lr_mae:,.2f}")

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_proc, y_train)
rf_pred = rf.predict(X_test_proc)
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f"Random Forest MAE:     ${rf_mae:,.2f}")
print(f"Improvement:           ${lr_mae - rf_mae:,.2f} ({(1 - rf_mae/lr_mae)*100:.1f}%)")


**What to notice:**
- Random Forest typically outperforms Linear Regression on this dataset because it captures non-linear relationships and feature interactions.
- The improvement in MAE shows how much those non-linear patterns matter.

> **Pitfall:** Random Forests can overfit if trees are too deep. Use `max_depth` or `min_samples_leaf` to control complexity.

---

## 5. Cross-Validation Comparison

A single train/test split is a coin flip. You might get a lucky split where the test set is easy, or an unlucky one where it is hard. Either way, one number is not enough to trust a model — you need to know how it performs across many different subsets of the data. This is where **cross-validation** comes in, and it is one of the most important ideas in applied machine learning.

The idea is elegant: instead of splitting the data once, we split it five times. Each time, we train on 4/5 of the data and test on the remaining 1/5, rotating which fifth is held out. This gives us five performance estimates — a mean and a standard deviation. The mean tells us the expected performance; the standard deviation tells us how much that performance varies across different data subsets. A model with high mean and low variance is reliable; a model with high mean and high variance is a gamble.

We use `cross_val_score` with 5-fold CV and the `neg_mean_absolute_error` scoring metric. The "neg" prefix is an sklearn convention: since cross-validation always maximises the score, and we want to *minimise* MAE, sklearn negates it so that "higher is better" still holds. Remember to negate the result when reporting.

The key question we are answering: is Random Forest's advantage over Linear Regression real, or just an artifact of one particular train/test split? Cross-validation gives us the honest answer.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv(r"housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)

lr_cv = cross_val_score(
    LinearRegression(), X_train_proc, y_train,
    cv=5, scoring="neg_mean_absolute_error"
)
rf_cv = cross_val_score(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_proc, y_train, cv=5, scoring="neg_mean_absolute_error"
)

print(f"LR  CV MAE: ${-lr_cv.mean():,.2f} (+/- ${lr_cv.std():,.2f})")
print(f"RF  CV MAE: ${-rf_cv.mean():,.2f} (+/- ${rf_cv.std():,.2f})")


**What to notice:**
- Cross-validation gives a mean and standard deviation across 5 folds.
- If RF's CV MAE is lower than LR's, the improvement is robust, not a fluke of one split.

> **Pitfall:** Use `neg_mean_absolute_error` (sklearn convention: higher is better, so MAE is negated). Remember to negate the result.

---

## 6. Feature Importance

A model that cannot be explained is a model that will not be trusted. In high-stakes applications — lending, healthcare, criminal justice — stakeholders will not accept "the computer said so." They want to know *why* the model makes its predictions. **Feature importance** is the most direct answer to that question.

Random Forest provides feature importances for free. The idea is intuitive: when building each tree, the algorithm chooses which feature to split on based on how much that split reduces impurity (variance for regression). Features that are used often and produce large reductions in impurity are more important. The result is a ranking: which features drive the model's predictions?

On California housing, the answer is no surprise: `median_income` dominates. This makes perfect sense — income is the strongest predictor of housing wealth, and the model has correctly identified it as the most informative feature. The one-hot encoded `ocean_proximity` columns and location features (longitude, latitude) also contribute, reflecting the old real estate adage: location, location, location.

But feature importance has a caveat: it is biased toward features with many unique values (high cardinality). A feature with 100 distinct values has more opportunities to produce a "good" split than a feature with only 2 values, even if the latter is more predictive. For a more reliable ranking, use **permutation importance**: shuffle one feature, measure how much the model's performance drops, and repeat for every feature. We will see this technique in later labs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv(r"housing.csv")
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    imputer.fit_transform(X_train.select_dtypes(include=[np.number])),
    columns=X_train.select_dtypes(include=[np.number]).columns,
    index=X_train.index,
)
X_train_cat = pd.get_dummies(X_train[["ocean_proximity"]], drop_first=True)
X_train_proc = pd.concat([X_train_num, X_train_cat], axis=1)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_proc, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train_proc.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance")
ax.set_title("Random Forest — Feature Importance (Housing)")
plt.tight_layout()
from IPython.display import display
display(plt.gcf())

print("Top 3 features:")
print(importances.sort_values(ascending=False).head(3))


**What to notice:**
- `median_income` is almost always the dominant predictor.
- `ocean_proximity` one-hot columns and location features (longitude/latitude) also contribute.
- Feature importance helps you explain the model to stakeholders.

> **Pitfall:** Feature importance from Random Forest can be biased toward high-cardinality features. Use permutation importance for a more reliable ranking.

---

## Exercises

Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell. It checks the logic of your function against small cases. A different implementation is fine. It does not require the same lines of code or an exact number. If a check fails, the lab page says what is wrong and what to change. It does not give the answer.

### Exercise 1 — Random forest vs linear regression MAE
Fit linear regression and a random forest on the processed housing features. Compare their test MAE.

**Follow-up:** Subtract the random forest test MAE from the linear regression test MAE. Return how much lower the forest error is. If the forest is not lower, the result is zero or negative.

<details>
<summary>Hint</summary>

`mean_absolute_error` on each model's test predictions, then linear-regression MAE minus random-forest MAE.

</details>

---


In [ ]:
def rf_mae_gap():
    """Subtract the random forest test MAE from the linear regression test MAE. Return how much lower the forest error is. If the forest is not lower, the result is zero or negative."""
    pass


In [ ]:
kaa_check()  # prints your answers so you can compare them with the expected outputs


### What to learn next
- Gradient boosting (XGBoost, LightGBM) for even better regression performance
- Hyperparameter tuning with GridSearchCV or RandomizedSearchCV
- Feature engineering: creating interaction terms and polynomial features

*Files in this folder: housing.csv.*


---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
